In [31]:
from dotenv import load_dotenv
import os
patkey = os.getenv("GITHUB_PAT")
google_key = os.getenv("google_key")
openai_key = os.getenv("openai_key")
load_dotenv()

True

In [21]:
from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp

params={
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {patkey}",
                "X-MCP-Toolsets": "default,actions",
            },
            "timeout": 30,
        }
name="GitHub MCP"

async def main(token):
    async with MCPServerStreamableHttp(
        name=name,
        params=params,
        cache_tools_list=True,
    ) as github_server:
        tools = await github_server.list_tools()

        for tool in tools:
            print(f"Name: {tool.name}")
            print(f"Description: {tool.description}")
            print("-" * 50)


res = await main(patkey)


Name: actions_get
Description: Get details about specific GitHub Actions resources.
Use this tool to get details about individual workflows, workflow runs, jobs, and artifacts by their unique IDs.

--------------------------------------------------
Name: actions_list
Description: Tools for listing GitHub Actions resources.
Use this tool to list workflows in a repository, or list workflow runs, jobs, and artifacts for a specific workflow or workflow run.

--------------------------------------------------
Name: actions_run_trigger
Description: Trigger GitHub Actions workflow operations, including running, re-running, cancelling workflow runs, and deleting workflow run logs.
--------------------------------------------------
Name: add_comment_to_pending_review
Description: Add review comment to the requester's latest pending pull request review. A pending review needs to already exist to call this (check with the user if not sure).
--------------------------------------------------
Name:

In [30]:
from agents import OpenAIChatCompletionsModel, set_tracing_export_api_key, AsyncOpenAI, Agent, Runner, trace

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
set_tracing_export_api_key(openai_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
async with MCPServerStreamableHttp(
    name=name,
    params=params
) as git_mcp:
    with trace("testing listing mcp"):
        agent = Agent(name="Investigator",instructions="You are an agent with ability to heal workflows",model=gemini_model, mcp_servers=[git_mcp] );
        result = await Runner.run(agent,input=""" avinash-verma-1998/MiniDI/actions/runs/38763716906
 what is status of run if it failed why did it fail""");
        print(result.final_output)

The GitHub Actions workflow run `38763716906` (in repository `avinash-verma-1998/MiniDI`) has a status of **completed** with a conclusion of **failure**.

### Why did it fail?
The failure occurred in the `build (Debug)` job because MSBuild could not find the project/solution file. Specifically, the error log shows:

```
MSBUILD : error MSB1009: Project file does not exist.
Switch: your-solution-name
```

It appears that the workflow file `.github/workflows/build.yml` contains placeholder environment variables (such as `Solution_Name: your-solution-name`) that were not updated with the correct file paths for your repository.

### How to fix it:
You need to update your workflow file `.github/workflows/build.yml` and replace the placeholder values with the actual file paths from your repository. 

For example, look for the `env` section in your workflow file and ensure the following are correctly set:
- `Solution_Name`: Should be the path to your `.sln` file (e.g., `MiniDI.sln`).
- `Test_